# Formalia

Please read the [assignment overview page](https://laura.alessandretti.com/comsocsci2026/wiki_pages/Assignments.html) carefully before proceeding. The page contains information about formatting (including formats etc), group sizes, and many other aspects of handing in the assignment. 

__If you fail to follow these simple instructions, it will negatively impact your grade!__

**Due date and time**: The assignment is due on Mar 3rd at 23:59. Hand in your Jupyter notebook file (with extension `.ipynb`) via DTU Learn _(Assignment 1)_. 

Remember to include in the first cell of your notebook:
* the link to your group's Git repository 
* group members' contributions


## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 
>
> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.
> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

## Part 2: Find Researchers using the OpenAlex API

> **Exercise 3: Find potential Computational Social Scientists** In this exercise, we'll use the OpenAlex API to compile a list of researchers in the field of Computational Social Science, focusing on those who have attended the IC2S2 conference in 2025. This will not only later on help you understand the landscape of Computational Social Science research but also develop practical skills in data collection and analysis. 
>
> Please read the text of the whole exercise before starting to work on it. 
>
> **Steps**
> 
> 1. **Retreive data.** Consider the set of unique researcher names that you collected in Week 1, Exercise 3. Use the _authors_ endpoint of the [OpenAlex API](https://docs.openalex.org/api-entities/authors) to _search_ these researchers in the database based on their names. Loop through the list and, for each researcher in your list, find: 
>     - their _id_: The OpenAlex ID for this author.
>     - their _display\_name_: The name of the author as a single string.
>     - their _works\_api\_url_: A URL that will get you a list of all this author's works.
>     - their _h\_index_ : The h-index for this author.
>     - their _works\_count_: The number of  Works this this author has created.
>     - their _country\_code_: The country code of their last known institution
> 2. **Data Storage** Store this information in a Pandas DataFrame and save it to file.
>
>    
> **Handling Challenges**
> 
> I expect that, while working on the steps above, you will encounter several obstacles. As you complete this exercise, you are expected to:     
>
>    - Identify problems that arise.      
>    - Improve your solutions to address such problems, making reasonable decisions when data is incomplete or ambiguous.       
>
> **Reflection Questions**
>  Answer the following questions __(max 150 words for each question)__: 
>
>    - Which challenges did you encounter? How did you address them?
>    - Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data? 
>      


In [1]:
import requests
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
import os

parent_dir = Path.cwd().parent
os.chdir(parent_dir)

BASE_URL = "https://api.openalex.org/authors"
API_KEY = os.environ.get("OPENALEX_API_KEY")

In [2]:
with open("author_names_filtered.txt", "r") as f:
    author_names = sorted({line.strip() for line in f if line.strip()})

print(f"Loaded {len(author_names)} unique author names")

Loaded 1581 unique author names


In [8]:
def format_author_result(item):
    return {
        "id": item.get("id"),
        "display_name": item.get("display_name"),
        "works_api_url": item.get("works_api_url"),
        "h_index": (item.get("summary_stats") or {}).get("h_index"),
        "works_count": item.get("works_count"),
        "country_code": (
            item["last_known_institutions"][-1].get("country_code")
            if item.get("last_known_institutions")
            else None
        ),
    }


def search_authors_by_name(name, api_key=None):
    params = {"search": name}
    if api_key:
        params["api_key"] = api_key

    response = requests.get(BASE_URL, params=params, timeout=20)
    response.raise_for_status()
    payload = response.json()
    results = payload["results"]
    if len(results) == 0:
        return []
    
    best = results[0]
    for author in response["results"]:
        if author["relevance_score"] > best["relevance_score"]:
            best = author
    

    return format_author_result(best)

In [9]:
author_records = []
failed_names = []

for name in tqdm(author_names, desc="Searching OpenAlex authors"):
    try:
        author_records.extend(search_authors_by_name(name, api_key=API_KEY or None))
    except Exception as e:
        print(f"Failed to search for author: {name}: {e}")

authors_df = pd.DataFrame(author_records)
authors_df = authors_df.drop_duplicates(subset=["id"]).reset_index(drop=True)
authors_df.to_csv("researchers.csv", index=False)

print(f"Researchers saved: {len(authors_df)}")
print(f"Failed name lookups: {len(failed_names)}")
authors_df.head()

Searching OpenAlex authors:   0%|          | 1/1581 [00:00<03:11,  8.23it/s]

Failed to search for author: Aakriti Kumar: 429 Client Error: Too Many Requests for url: https://api.openalex.org/authors?search=Aakriti+Kumar&api_key=dOuehTDKplNtKzbWNlmhnM


Searching OpenAlex authors:   0%|          | 3/1581 [00:00<04:15,  6.18it/s]

Failed to search for author: Aaron Clauset: 429 Client Error: Too Many Requests for url: https://api.openalex.org/authors?search=Aaron+Clauset&api_key=dOuehTDKplNtKzbWNlmhnM
Failed to search for author: Aaron D Nichols: 429 Client Error: Too Many Requests for url: https://api.openalex.org/authors?search=Aaron+D+Nichols&api_key=dOuehTDKplNtKzbWNlmhnM


Searching OpenAlex authors:   0%|          | 4/1581 [00:00<03:58,  6.61it/s]

...

Failed to search for author: Abhisek Dash: 429 Client Error: Too Many Requests for url: https://api.openalex.org/authors?search=Abhisek+Dash&api_key=dOuehTDKplNtKzbWNlmhnM


Searching OpenAlex authors:   1%|          | 9/1581 [00:01<05:14,  5.00it/s]

Failed to search for author: Abraham Israeli: 429 Client Error: Too Many Requests for url: https://api.openalex.org/authors?search=Abraham+Israeli&api_key=dOuehTDKplNtKzbWNlmhnM


Searching OpenAlex authors:   1%|          | 9/1581 [00:02<06:02,  4.34it/s]


KeyboardInterrupt: 

## Part 3: Collect Research Articles

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
>In this exercise, we'll leverage the OpenAlex API to gather information on research articles authored by participants of the IC2S2 2025 conference, referred to as *IC2S2 authors*. **Before you start, please ensure you read through the entire exercise.**
>
> 
> **Steps:**
>  
> 1. **Retrieve Data:** Start with the dataset of *IC2S2 authors* you collected in Week 2, Exercise 3 (called dataset D1 in the figure above). Use the OpenAlex API [works endpoint](https://docs.openalex.org/api-entities/works) to fetch their research articles. For each article, retrieve the following details:
>    - _id_: The unique OpenAlex ID for the work.
>    - _publication_year_: The year the work was published.
>    - _cited_by_count_: The number of times the work has been cited by other works.
>    - _author_ids_: The OpenAlex IDs for the authors of the work.
>    - _title_: The title of the work.
>    - _abstract_inverted_index_: The abstract of the work, formatted as an inverted index.
> 
>     **Important Note on Paging:** By default, the OpenAlex API limits responses to 25 works per request. For more efficient data retrieval, I suggest to adjust this limit to 200 works per request. Even with this adjustment, you will need to implement pagination to access all available works for a given query. This ensures you can systematically retrieve the complete set of works beyond the initial 200. Find guidance on implementing pagination [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/paging#cursor-paging).
>
> 2. **Data Storage:** Organize the retrieved information into two Pandas DataFrames and save them to two files in a suitable format:
>    - Dataset D2: The *IC2S2 papers* dataset should include: *id, publication\_year, cited\_by\_count, author\_ids*.
>    - Dataset D3: The *IC2S2 abstracts* dataset should include: *id, title, abstract\_inverted\_index*.
>  
>
> **Filters:**
> To ensure the data we collect is relevant and manageable, apply the following filters:
>     
>    - Only include *IC2S2 authors* with a total work count between 5 and 5,000.    
>    - Retrieve only works that have received more than 10 citations.    
>    - Limit to works authored by fewer than 10 individuals.    
>    - Include only works relevant to Computational Social Science (focusing on: Sociology OR Psychology OR Economics OR Political Science) AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science), as defined by their [Concepts](https://docs.openalex.org/api-entities/works/work-object#concepts). *Note*: here we only consider Concepts at *level=0* (the most coarse definition of concepts).     
>
> **Efficiency Tips:**
> Writing efficient code in this exercise is **crucial**. To speed up your process:
> 
> - **Apply filters directly in your request:** When possible, use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) of the *works* endpoint to apply the filters above directly in your API request, ensuring only relevant data is returned. Learn about combining multiple filters [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).  
> - **Bulk requests:** Instead of sending one request for each author, you can use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) to query works by multiple authors in a single request. *Note: My testing suggests that can only include up to 25 authors per request.*
> - **Use multiprocessing:** Implement multiprocessing to handle multiple requests simultaneously. I highly recommmend [Joblib’s Parallel](https://joblib.readthedocs.io/en/stable/) function for that, and [tqdm](https://tqdm.github.io/) can help monitor progress of your jobs. Remember to stay within [the rate limit](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication) of 100 requests per second.
>
>
>   
> For reference, employing these strategies allowed me to fetch the data in about 30 seconds using 5 cores on my laptop. I obtained a dataset of approximately 25 MB (including both the *IC2S2 abstracts* and *IC2S2 papers* files).
> 
>
> **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__: 
> 
> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     
> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?    
> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?    

In [5]:
import json
from itertools import islice

WORKS_URL = "https://api.openalex.org/works"
CONCEPTS_URL = "https://api.openalex.org/concepts"

authors_df = pd.read_csv("researchers.csv")

researchers_filtered = authors_df[
    (authors_df["works_count"] >= 5) &
    (authors_df["works_count"] <= 5000)
].copy()

author_ids = researchers_filtered["id"].dropna().drop_duplicates().tolist()
print(f"IC2S2 authors (5-5000 works): {len(author_ids)}")

IC2S2 authors (5-5000 works): 4215


In [13]:
import time

CSS = ["Sociology", "Psychology", "Economics", "Political Science"]
QD = ["Mathematics", "Physics", "Computer Science"]


def concept_id(concept_name):
    params = {"search": concept_name}
    if API_KEY:
        params["api_key"] = API_KEY
    response = requests.get(CONCEPTS_URL, params=params, timeout=20)
    response.raise_for_status()
    return response.json()["results"][0]["id"]

if False:
    CSS_IDS = [concept_id(name) for name in CSS[:1]]
    QD_IDS = [concept_id(name) for name in QD]
else:
    SOC = "C144133560"
    PSY = "C15744967"
    ECO = "C162324750"
    POL = "C117744445"

    MATH = "C33923547"
    PHYS = "C121332964"
    CS = "C41008148"

    CSS_IDS = [SOC, PSY, ECO, POL]
    QD_IDS = [MATH, PHYS, CS]

print("CSS concept IDs:", CSS_IDS)
print("QD concept IDs:", QD_IDS)

CSS concept IDs: ['C144133560', 'C15744967', 'C162324750', 'C117744445']
QD concept IDs: ['C33923547', 'C121332964', 'C41008148']


In [19]:
def chunks(values, size):
    iterator = iter(values)
    while True:
        batch = list(islice(iterator, size))
        if not batch:
            return
        yield batch


def work_matches_concepts(work, css_ids, qd_ids):
    level0_ids = {
        concept["id"]
        for concept in work.get("concepts", [])
        if concept.get("level") == 0
    }
    has_css = any(cid in level0_ids for cid in css_ids)
    has_qd = any(cid in level0_ids for cid in qd_ids)
    return has_css and has_qd


def fetch_works_for_author_batch(batch_author_ids, css_ids, qd_ids, api_key=None):
    all_batch_works = []
    cursor = "*"

    while True:
        params = {
            "filter": ",".join([
                f"authorships.author.id:{'|'.join(batch_author_ids)}",
                "cited_by_count:>10",
                "authors_count:<10",
                f"concepts.id:{'|'.join(css_ids)}",
                f"concepts.id:{'|'.join(qd_ids)}"
            ]),
            "per_page": 200,
            "cursor": cursor,
        }
        if api_key:
            params["api_key"] = api_key

        response = requests.get(WORKS_URL, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()

        results = payload.get("results", [])
        if not results:
            break

        for work in results:
            if not work_matches_concepts(work, css_ids, qd_ids):
                continue

            all_batch_works.append({
                "id": work.get("id"),
                "publication_year": work.get("publication_year"),
                "cited_by_count": work.get("cited_by_count"),
                "author_ids": [a["author"]["id"] for a in work.get("authorships", []) if a.get("author")],
                "title": work.get("title"),
                "abstract_inverted_index": work.get("abstract_inverted_index"),
            })

        next_cursor = (payload.get("meta") or {}).get("next_cursor")
        if not next_cursor:
            break
        cursor = next_cursor

    return all_batch_works

In [ ]:
from joblib import Parallel, delayed

batch_list = [
    author_ids[i:i + 25] for i in range(0, len(author_ids), 25)
]

all_results = Parallel(n_jobs=4)(
    delayed(fetch_works_for_author_batch)(batch, CSS_IDS, QD_IDS, api_key=API_KEY or None)
    for batch in tqdm(batch_list, desc="Fetching works for author batches")
)

all_works = [work for batch_works in all_results for work in batch_works]

works_df = pd.DataFrame(all_works).drop_duplicates(subset=["id"]).reset_index(drop=True)
print(f"Unique works collected: {len(works_df)}")
works_df.head()

HTTPError: 429 Client Error: Too Many Requests for url: https://api.openalex.org/works?filter=authorships.author.id%3Ahttps%3A%2F%2Fopenalex.org%2FA5103136756%7Chttps%3A%2F%2Fopenalex.org%2FA5050463790%7Chttps%3A%2F%2Fopenalex.org%2FA5110548648%7Chttps%3A%2F%2Fopenalex.org%2FA5029894607%7Chttps%3A%2F%2Fopenalex.org%2FA5102294064%7Chttps%3A%2F%2Fopenalex.org%2FA5077307812%7Chttps%3A%2F%2Fopenalex.org%2FA5036743197%7Chttps%3A%2F%2Fopenalex.org%2FA5094706224%7Chttps%3A%2F%2Fopenalex.org%2FA5056193227%7Chttps%3A%2F%2Fopenalex.org%2FA5055236010%7Chttps%3A%2F%2Fopenalex.org%2FA5004420568%7Chttps%3A%2F%2Fopenalex.org%2FA5121506531%7Chttps%3A%2F%2Fopenalex.org%2FA5058731900%7Chttps%3A%2F%2Fopenalex.org%2FA5027136376%7Chttps%3A%2F%2Fopenalex.org%2FA5008132739%7Chttps%3A%2F%2Fopenalex.org%2FA5034647482%7Chttps%3A%2F%2Fopenalex.org%2FA5030915311%7Chttps%3A%2F%2Fopenalex.org%2FA5032150674%7Chttps%3A%2F%2Fopenalex.org%2FA5028325693%7Chttps%3A%2F%2Fopenalex.org%2FA5031654351%7Chttps%3A%2F%2Fopenalex

In [28]:
D2 = pd.DataFrame({
    "id": works_df["id"],
    "publication_year": works_df["publication_year"],
    "cited_by_count": works_df["cited_by_count"],
    "author_ids": works_df["author_ids"].apply(json.dumps),
})

D3 = pd.DataFrame({
    "id": works_df["id"],
    "title": works_df["title"],
    "abstract_inverted_index": works_df["abstract_inverted_index"].apply(
        lambda value: json.dumps(value) if isinstance(value, dict) else None
    ),
})

D2.to_csv("ic2s2_papers_D2.csv", index=False)
D3.to_csv("ic2s2_abstracts_D3.csv", index=False)

print(f"Saved D2 rows: {len(D2)}")
print(f"Saved D3 rows: {len(D3)}")

Saved D2 rows: 41069
Saved D3 rows: 41069


In [29]:
n_works = len(D2)
unique_researchers = (
    D2["author_ids"]
    .apply(json.loads)
    .explode()
    .dropna()
    .nunique()
)

print(f"Dataset summary: {n_works} works, {unique_researchers} unique co-authors")

Dataset summary: 41069 works, 79204 unique co-authors
